In [0]:
%python
df=spark.read.format("csv").option("header","True").load(/Volumes/workspace/default/datasetsoforders/List of Orders.csv)

In [0]:
%python

data = [

(1, "C001", "Laptop", "Electronics", "50000", "2", "2024-01-10"),
(2, "C002", "Shoes", "Fashion", "2000", "1", "2024-01-11"),
(3, "C001", "Mouse", "Electronics", "abc", "1", "2024-01-12"),
(4, "C003", "T-shirt", "Fashion", "1500", None, "2024-01-13"),
(5, "C004", "Mobile", "Electronics", "30000", "1", "2024-01-14"),
(6, "C002", "Shoes", "Fashion", "2000", "1", "2024-01-11"),

]
columns=["Order_id","Customer_id","Product","Category","Price","Quantity","Order_date"]
df_raw=spark.createDataFrame(data,columns)

df_raw.show()

In [0]:
df_raw.printSchema()
## not good data with datatyoe not good,duplicates,null data

In [0]:
%python


#Data cleaning process

from pyspark.sql.functions import col

df_clean=df_raw \
    .dropDuplicates() \
        .withColumn("price",col("price").cast("int")) \
            .withColumn("quantity",col("quantity").cast("int")) \
                .fillna({"quantity":1}).show()

    #drop duplicates removes the duplicate record
    #withColum.... price into integer using case change
    #fillna  will fill your null value with any value which u have given



In [0]:
from pyspark.sql.functions import expr,col

df_clean=df_raw \
    .dropDuplicates() \
        .withColumn("price",expr("try_cast(price as int)")) \
            .withColumn("quantity",col("quantity").cast("int")) \
                .fillna({"quantity":1})

display(df_clean)              

In [0]:
%python

#senario 2 to create a business column 

df_clean = df_clean.withColumn(
    "total_amount",col("price")*col("quantity")

)
df_clean.show()

In [0]:
#selectExpr
df_clean.selectExpr("sum(total_amount)as total_revenue")

In [0]:
%python

df_clean.filter(col("price").isNull()).show()

In [0]:
#path = /Volumes/workspace/default/datasetsoforders/netflix_titles.csv

In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
%python
spark =SparkSession.builder \
    .appName("netflix") \
        .getOrCreate() 


In [0]:
df_practice = spark.read \
    .format("csv") \
    .option("header","True") \
    .option("inferSchema","True") \
    .load("/Volumes/workspace/default/datasetsoforders/netflix_titles.csv")

In [0]:
display(df_practice)

In [0]:
df_practice.printSchema()

In [0]:
#find the top 10 countries with most contents
df_practice.groupBy("country") \
    .count() \
        .orderBy(col("count").desc()) \
            .show(10)


             

In [0]:
#top 5 director with most content

df_practice.groupBy("director") \
    .count() \
        .orderBy(col("count").desc()) \
            .show(5)

In [0]:
df_practice.printSchema()

In [0]:
#convert date added to proper date
    #extrack numeric values from duration 90 minutes -90(numeric)
df_clean=df_practice \
   .withColumn(
        "date_added",
        to_date(col("date_added"), "mmmm-dddd-yyyy")
    ) \
    .withColumn("duration_int",
                regexp_extract(col("duration"),"\\d+",0).cast("int")
                )
    

    #for eg if your date is september 10,2021 which we are converting into 
    #\\d+ for eg:90 min -90

In [0]:
df_clean.printSchema()

In [0]:
df_clean.select(
    "title",
    "date_added",
    "duration_int"
)

In [0]:
df_practice.select([
 sum(col(c).isNull().cast("int")).alias(c)
 for c in df_practice.columns
]).show()



#col(c).isNUll will return True/False for each row

# cast("int") will convert into True- make it into 1 and False- make it to 0.

#sum will give all the result

In [0]:
#remove rows with null values -dropna
df_no_nulls=df_practice.dropna()
df_no_nulls.show()

In [0]:
df_filtered=df_practice.dropna(
    subset=["director","country"]

)
df_filtered.show()

In [0]:
#remove row where all values are null

#df_clean=df_practice.dropna(how="all").   use this to delete 
 
df_filled=df_practice.fillna({
    "director" : "unknown",
    "country"  : "not-availaible",
    "release_year" : 0
})